# Assignment 3: Content-review data contract

**Lane 2: Refresh / Content Opportunity Scoring.** Earlier search measurements help
us investigate later click decline as a proxy for review priority. This exercise does
not establish that refreshing a page causes recovery.

Run locally with the **Python (FlyRank local)** kernel. From the repository root:

```powershell
python -m venv .venv
.venv/Scripts/python.exe -m pip install -r requirements-notebooks.txt
.venv/Scripts/python.exe -m ipykernel install --prefix .venv --name flyrank-local --display-name "Python (FlyRank local)"
```

Request access at [the warehouse](https://huggingface.co/datasets/FlyRank/internship-warehouse).
Open the Git-ignored `.env` file at the repository root and put your Read token after
`HF_TOKEN=`. The notebook loads that file automatically. An existing process environment
variable takes precedence. After changing `.env`, restart the kernel and Run All.

Launch Jupyter from the repository root:

```powershell
.venv/Scripts/python.exe -m jupyterlab
```

Never paste a token into source, output, or Git. The first authenticated run reads March
only and caches selected fields in ignored `work/outputs/w03_data_contract/`. Subsequent
runs use that local cache. Internet is required for the first run. Restart the kernel and
Run All before submission. Keep `.env` local; it must never be committed.

Sources: [repo lane guide](../../docs/ml-intern-dataset-and-lane-guide.md),
[data rules](../../skills/flyrank/flyrank-data/SKILL.md), and the warehouse release card.

## 1. Unit of analysis + time window

1. **One row:** raw data is one `report_date × client_hash_id × content_hash_id`.
   The feature frame is one client/content pair at the March 16 decision date.
2. **Tables:** `fact_content_daily_performance` provides search measurements;
   `dim_clients` provides `gsc_data_start`. No content or query-level join is needed.
3. **Windows:** features use March 1–12, 2026; outcomes use March 16–27, 2026.
   Each is 12 days. March 13–15 is a three-day reporting buffer, an assumption rather
   than evidence that every historical record was finalized then. June stays sealed.
4. **Label:** `is_declining = outcome_clicks <= 0.8 * total_clicks`, meaning at least a
   20% decline. This ranks risk of observed decline, not the benefit of editing.
5. **Exclusion:** future outcome measurements and label-derived columns cannot enter
   the honest model. IDs are context, never predictors.

Eligibility requires 12 usable daily observations in each window, at least 20 feature-window
clicks, and positive feature-window impressions. Twenty clicks and 20% decline are exercise
policies, not validated business thresholds. Outcome coverage determines retrospective
evaluation eligibility; it would not be known when issuing a real recommendation.

## 2. Fields: feature / label / context / excluded

| Role | Fields and use |
|---|---|
| Features | `total_impressions`, `total_clicks`, `ctr`, `weighted_position`, `click_change` |
| Label | `is_declining`, calculated separately from `outcome_clicks` and earlier clicks |
| Context | IDs, dates, tracking start, availability, observation counts, release revision |
| Excluded | `outcome_clicks`, deliberate `leaked_label`, future-window measurements, private text |

Raw `gsc_impressions`, `gsc_clicks`, and `gsc_avg_position` are feature ingredients **only
inside the feature window**. Outcome clicks are label ingredients. Earlier click totals
also define the comparison baseline; they are legitimate earlier information, unlike future clicks.
Nulls are not zero activity. CTR is a fraction, not a percentage. Position values at or below
zero, nonfinite values, or observations with no impressions do not enter weighted position.
Missing aggregate position is imputed using the training median only.

In [1]:
import os
import json
import platform
from pathlib import Path
from importlib.metadata import version
from dotenv import load_dotenv
import duckdb
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'skills/README.md').is_file()), None)
if ROOT is None:
    raise RuntimeError('Launch this notebook from inside the project repository.')
load_dotenv(ROOT / '.env', override=False)
CACHE = ROOT / 'work/outputs/w03_data_contract'
CACHE.mkdir(parents=True, exist_ok=True)
DB_PATH = CACHE / 'march-v1.duckdb'
MANIFEST_PATH = CACHE / 'march-v1.json'
FEATURE_START, FEATURE_END = '2026-03-01', '2026-03-12'
DECISION_DATE = '2026-03-16'
OUTCOME_START, OUTCOME_END = '2026-03-16', '2026-03-27'
FEATURE_COLS = ['total_impressions', 'total_clicks', 'ctr', 'weighted_position', 'click_change']
assert FEATURE_END < DECISION_DATE <= OUTCOME_START
assert (pd.Timestamp(OUTCOME_END) - pd.Timestamp(OUTCOME_START)).days == 11
assert (pd.Timestamp(FEATURE_END) - pd.Timestamp(FEATURE_START)).days == 11
print('Python:', platform.python_version())
display(pd.Series({p: version(p) for p in ['duckdb', 'pandas', 'numpy', 'scikit-learn']}, name='version'))

Python: 3.13.6


duckdb          1.5.5
pandas          3.0.6
numpy           2.5.3
scikit-learn    1.9.1
Name: version, dtype: str

### Local cache setup and schema inspection (not verification queries)

The cache records the dataset revision, selected source files, and actual schemas. Only
March Parquet files and the client dimension are read. A published `gsc_data_available`
Boolean field, if present, is combined with history and metric validity checks. Otherwise
`search_data_available` is explicitly derived from those checks. This operational fallback
cannot prove that tracking was uninterrupted. Analytics availability is irrelevant here.

In [2]:
if 'con' in globals():
    con.close()
con = duckdb.connect(str(DB_PATH))
con.execute("SET memory_limit = '1GB'")
con.execute('SET threads = 2')

def require_fields(schema, required, table):
    if not set(required).issubset(schema):
        raise RuntimeError(f'Required fields missing in {table}; inspect the release schema before continuing.')

if not MANIFEST_PATH.exists():
    token = os.environ.get('HF_TOKEN', '').strip()
    if not token:
        raise RuntimeError('HF_TOKEN is missing. Accept dataset access, set HF_TOKEN in the project .env file or launch environment, then restart the kernel. No warehouse results have been produced.')
    from huggingface_hub import HfApi
    try:
        info = HfApi(token=token).dataset_info('FlyRank/internship-warehouse', files_metadata=False)
        revision = info.sha
        paths = sorted(s.rfilename for s in info.siblings
                       if s.rfilename.startswith('fact_content_daily_performance/month=2026-03/')
                       and s.rfilename.endswith('.parquet'))
        if not paths:
            raise ValueError('No March partitions')
        # SQL is never printed. The in-memory secret is not persisted to the cache.
        escaped_token = token.replace("'", "''")
        con.execute(f"CREATE OR REPLACE SECRET w03_hf (TYPE huggingface, TOKEN '{escaped_token}')")
        base = f'hf://datasets/FlyRank/internship-warehouse@{revision}/'
        con.read_parquet([base + p for p in paths]).create_view('remote_march', replace=True)
        con.read_parquet(base + 'dim_clients.parquet').create_view('remote_clients', replace=True)
        daily_schema = {r[0]: r[1] for r in con.execute('DESCRIBE remote_march').fetchall()}
        client_schema = {r[0]: r[1] for r in con.execute('DESCRIBE remote_clients').fetchall()}
        required = ['report_date', 'client_hash_id', 'content_hash_id',
                    'gsc_impressions', 'gsc_clicks', 'gsc_avg_position']
        require_fields(daily_schema, required, 'daily performance')
        require_fields(client_schema, ['client_hash_id', 'gsc_data_start'], 'clients')
        has_flag = 'gsc_data_available' in daily_schema
        if has_flag and daily_schema['gsc_data_available'] != 'BOOLEAN':
            raise ValueError('Search availability flag is not Boolean')
        selected = required + (['gsc_data_available'] if has_flag else [])
        con.execute('BEGIN')
        con.execute('CREATE OR REPLACE TABLE march_raw AS SELECT ' + ', '.join(selected) + ' FROM remote_march')
        con.execute('CREATE OR REPLACE TABLE clients AS SELECT client_hash_id, gsc_data_start FROM remote_clients')
        con.execute('COMMIT')
        manifest = {'cache_version': 1, 'dataset': 'FlyRank/internship-warehouse',
                    'revision': revision, 'month': '2026-03', 'source_files': paths,
                    'daily_schema': daily_schema, 'client_schema': client_schema,
                    'has_search_flag': has_flag}
        MANIFEST_PATH.write_text(json.dumps(manifest, indent=2), encoding='utf-8')
    except Exception:
        con.close()
        raise RuntimeError('Warehouse setup failed. Check gated-dataset approval, Read-token permissions, connectivity, and release schema. Credentials and remote error details are suppressed. No results should be claimed.') from None
    finally:
        token = escaped_token = None

manifest = json.loads(MANIFEST_PATH.read_text(encoding='utf-8'))
assert manifest['cache_version'] == 1 and manifest['month'] == '2026-03'
assert all('/month=2026-03/' in p for p in manifest['source_files'])
client_keys = con.execute('SELECT client_hash_id FROM clients').df()
assert client_keys.client_hash_id.notna().all() and client_keys.client_hash_id.is_unique, 'Client dimension key is invalid.'
flag_check = 'f.gsc_data_available IS TRUE AND' if manifest['has_search_flag'] else ''
con.execute(f"""
CREATE OR REPLACE TEMP VIEW march AS
SELECT f.*, CAST(f.report_date AS DATE) AS day, c.gsc_data_start,
       ({flag_check}
        c.gsc_data_start IS NOT NULL AND CAST(c.gsc_data_start AS DATE) <= CAST(f.report_date AS DATE)
        AND f.gsc_clicks IS NOT NULL AND isfinite(f.gsc_clicks) AND f.gsc_clicks >= 0
        AND f.gsc_impressions IS NOT NULL AND isfinite(f.gsc_impressions) AND f.gsc_impressions >= 0
        AND f.gsc_clicks <= f.gsc_impressions) AS search_data_available
FROM march_raw f LEFT JOIN clients c USING (client_hash_id)
""")
print('Cached release revision:', manifest['revision'])
print('Availability:', 'published search flag plus validity checks' if manifest['has_search_flag'] else 'derived from tracking start and metric validity; no published search flag')
display(pd.DataFrame([(k, v) for k, v in manifest['daily_schema'].items()
                      if k in ['report_date', 'client_hash_id', 'content_hash_id', 'gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'gsc_data_available']],
                     columns=['required_or_availability_field', 'source_type']))

Cached release revision: 50cbf7c3909d07be4d1b5906b4d09e882e5acbf2
Availability: published search flag plus validity checks


,required_or_availability_field,source_type
0,report_date,DATE
1,client_hash_id,VARCHAR
2,content_hash_id,VARCHAR
3,gsc_data_available,BOOLEAN
4,gsc_impressions,BIGINT
5,gsc_clicks,BIGINT
6,gsc_avg_position,DOUBLE


## 3. Verify it with queries, then build features and test leakage

### Verification query 1: raw grain

Zero duplicate groups supports the daily client/content grain. Null keys also fail the check.

In [3]:
grain_sql = """
WITH groups AS (
    SELECT day, client_hash_id, content_hash_id, COUNT(*) AS n
    FROM march GROUP BY 1, 2, 3
)
SELECT COUNT(*) FILTER (WHERE n > 1) AS duplicate_groups,
       COALESCE(SUM(n - 1) FILTER (WHERE n > 1), 0) AS extra_rows,
       COUNT(*) FILTER (WHERE day IS NULL OR client_hash_id IS NULL OR content_hash_id IS NULL) AS null_key_groups
FROM groups
"""
grain_result = con.execute(grain_sql).df()
display(grain_result)
assert (grain_result.iloc[0] == 0).all(), 'Raw grain failed. Investigate rather than deduplicate silently.'

,duplicate_groups,extra_rows,null_key_groups
0,0,0.0,0


### Verification query 2: row count and date span

In [4]:
count_sql = """
SELECT COUNT(*) AS row_count, COUNT(DISTINCT client_hash_id) AS clients,
       COUNT(DISTINCT (client_hash_id, content_hash_id)) AS content_items,
       MIN(day) AS first_date, MAX(day) AS last_date
FROM march
"""
count_result = con.execute(count_sql).df()
display(count_result)
assert count_result.row_count.iloc[0] > 0, 'The month is empty.'
assert pd.Timestamp('2026-03-01') <= count_result.first_date.iloc[0]
assert count_result.last_date.iloc[0] <= pd.Timestamp('2026-03-31')

,row_count,clients,content_items,first_date,last_date
0,9841378,55,331437,2026-03-01,2026-03-31


### Verification query 3: availability with `IS TRUE`

In [5]:
availability_sql = """
SELECT COUNT(*) AS rows_before,
       COUNT(*) FILTER (WHERE search_data_available IS TRUE) AS rows_after,
       COUNT(*) - COUNT(*) FILTER (WHERE search_data_available IS TRUE) AS rows_excluded,
       COUNT(*) FILTER (WHERE search_data_available IS NULL) AS rows_unknown
FROM march
"""
availability_result = con.execute(availability_sql).df()
display(availability_result)
assert availability_result.rows_after.iloc[0] > 0, 'No usable search observations survive.'

,rows_before,rows_after,rows_excluded,rows_unknown
0,9841378,3611061,6230317,0


### Feature extraction (separate from the three verification queries)

| Feature | Available when? |
|---|---|
| Total impressions | Knowable at the decision moment because it uses March 1–12 exposure, with an assumed three-day reporting buffer. |
| Total clicks | Knowable at the decision moment because it uses only clicks recorded March 1–12, under the reporting-delay assumption. |
| CTR | Knowable at the decision moment because both totals come from March 1–12; it is a fraction. |
| Weighted position | Knowable at the decision moment because position and its impression weights come only from March 1–12. |
| Click change | Knowable at the decision moment because both six-day subwindows end before March 16. |

Click change is `(March 7–12 clicks - March 1–6 clicks) / max(March 1–6 clicks, 1)`.
The denominator floor handles zero earlier clicks; it is not a conventional percentage
change when that baseline is zero. Missing position is retained for train-only imputation.

In [6]:
feature_sql = f"""
WITH earlier AS (
 SELECT client_hash_id, content_hash_id, COUNT(*) AS feature_days,
        SUM(gsc_impressions)::DOUBLE AS total_impressions,
        SUM(gsc_clicks)::DOUBLE AS total_clicks,
        SUM(CASE WHEN gsc_avg_position > 0 AND isfinite(gsc_avg_position) AND gsc_impressions > 0
                 THEN gsc_avg_position * gsc_impressions END)
        / NULLIF(SUM(CASE WHEN gsc_avg_position > 0 AND isfinite(gsc_avg_position) AND gsc_impressions > 0
                         THEN gsc_impressions END), 0) AS weighted_position,
        SUM(CASE WHEN day <= DATE '2026-03-06' THEN gsc_clicks ELSE 0 END)::DOUBLE AS first_clicks,
        SUM(CASE WHEN day >= DATE '2026-03-07' THEN gsc_clicks ELSE 0 END)::DOUBLE AS later_clicks
 FROM march WHERE search_data_available IS TRUE
 AND day BETWEEN DATE '{FEATURE_START}' AND DATE '{FEATURE_END}' GROUP BY 1, 2
), outcomes AS (
 SELECT client_hash_id, content_hash_id, COUNT(*) AS outcome_days,
        SUM(gsc_clicks)::DOUBLE AS outcome_clicks
 FROM march WHERE search_data_available IS TRUE
 AND day BETWEEN DATE '{OUTCOME_START}' AND DATE '{OUTCOME_END}' GROUP BY 1, 2
)
SELECT e.*, o.outcome_days, o.outcome_clicks,
       total_clicks / NULLIF(total_impressions, 0) AS ctr,
       (later_clicks - first_clicks) / GREATEST(first_clicks, 1) AS click_change
FROM earlier e LEFT JOIN outcomes o USING (client_hash_id, content_hash_id)
"""
aggregates = con.execute(feature_sql).df()
complete = (aggregates.feature_days == 12) & (aggregates.outcome_days == 12)
eligible = complete & (aggregates.total_clicks >= 20) & (aggregates.total_impressions > 0)
display(pd.DataFrame({'stage': ['Has usable feature observations', 'Complete feature and outcome windows', 'Passes volume thresholds'],
                      'content_items': [len(aggregates), int(complete.sum()), int(eligible.sum())]}))
data = aggregates.loc[eligible].copy().reset_index(drop=True)
if data.empty:
    raise RuntimeError('No eligible pages. Inspect coverage and thresholds; do not invent results or change the label to chase a score.')
assert not data.duplicated(['client_hash_id', 'content_hash_id']).any()
X = data[FEATURE_COLS].copy()
assert X.shape[1] == 5 and not np.isinf(X.to_numpy()).any()
assert X.drop(columns='weighted_position').notna().all().all()
print('Feature frame shape:', X.shape)
display(data[['client_hash_id', 'content_hash_id'] + FEATURE_COLS].head())
display(X.isna().sum().rename('missing_rows').to_frame())
data['is_declining'] = (data.outcome_clicks <= 0.8 * data.total_clicks).astype(int)
y = data['is_declining'].copy()
print('Label: at least 20% fewer clicks in the later 12-day window.')
display(y.value_counts().sort_index().rename_axis('is_declining').to_frame('rows'))

,stage,content_items
0,Has usable feature observations,146810
1,Complete feature and outcome windows,64356
2,Passes volume thresholds,2888


Feature frame shape: (2888, 5)


,client_hash_id,content_hash_id,total_impressions,total_clicks,ctr,weighted_position,click_change
0,client_62f4a7e64f5e0096,content_77ffcdc7ab5e71f5,4804.0,57.0,0.011865,1.501041,-0.272727
1,client_73cda7b4e4f265ea,content_252aa5480bb1f8d7,30716.0,24.0,0.000781,2.235675,0.181818
2,client_73cda7b4e4f265ea,content_baad8e51d439b6b5,4598.0,25.0,0.005437,4.922793,0.272727
3,client_73cda7b4e4f265ea,content_07becf910983f0c1,6693.0,29.0,0.004333,3.358882,-0.388889
4,client_73cda7b4e4f265ea,content_fd65e0671914dc2c,2863.0,21.0,0.007335,4.100594,1.000000


,missing_rows
total_impressions,0
total_clicks,0
ctr,0
weighted_position,0
click_change,0


Label: at least 20% fewer clicks in the later 12-day window.


,rows
is_declining,
0,1542
1,1346


### Honest score, deliberate leak, and removal

Hold out 25% of clients with seed 42. This tests unseen clients in the same period, not
generalization to future months. Report balanced accuracy because classes may be unequal.
The deliberately leaked column copies the answer exactly. A near-perfect leaking score
demonstrates circularity, not model skill. All comparisons use identical rows and splits.

In [7]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import balanced_accuracy_score

if data.client_hash_id.nunique() < 2 or y.nunique() != 2:
    raise RuntimeError('Evaluation requires at least two clients and both label classes.')
train_idx, test_idx = next(GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
                          .split(X, y, groups=data.client_hash_id))
train_clients = set(data.iloc[train_idx].client_hash_id)
test_clients = set(data.iloc[test_idx].client_hash_id)
assert train_clients.isdisjoint(test_clients)
if y.iloc[train_idx].nunique() != 2 or y.iloc[test_idx].nunique() != 2:
    raise RuntimeError('The fixed client split lacks a label class. Report this limitation; do not hunt for a favorable seed.')
if X.iloc[train_idx].weighted_position.notna().sum() == 0:
    raise RuntimeError('Training position is entirely missing; median imputation is undefined. Revisit the contract.')
display(pd.DataFrame({'split': ['train', 'test'], 'clients': [len(train_clients), len(test_clients)],
                      'rows': [len(train_idx), len(test_idx)],
                      'positive_rows': [int(y.iloc[train_idx].sum()), int(y.iloc[test_idx].sum())]}))

def fit_tree(frame):
    model = make_pipeline(SimpleImputer(strategy='median'),
                          DecisionTreeClassifier(max_depth=3, random_state=42))
    model.fit(frame.iloc[train_idx], y.iloc[train_idx])
    score = balanced_accuracy_score(y.iloc[test_idx], model.predict(frame.iloc[test_idx]))
    return model, score

baseline = DummyClassifier(strategy='most_frequent').fit(X.iloc[train_idx], y.iloc[train_idx])
baseline_score = balanced_accuracy_score(y.iloc[test_idx], baseline.predict(X.iloc[test_idx]))
honest_model, honest_score = fit_tree(X)
X_leaky = X.assign(leaked_label=y)
leaky_model, leaky_score = fit_tree(X_leaky)
display(pd.DataFrame({'experiment': ['Majority baseline', 'Honest five features', 'Deliberate label copy'],
                      'balanced_accuracy': [baseline_score, honest_score, leaky_score]}))
print(f'Leak score change relative to honest model: {leaky_score - honest_score:+.3f}')

,split,clients,rows,positive_rows
0,train,16,1839,940
1,test,6,1049,406


,experiment,balanced_accuracy
0,Majority baseline,0.500000
1,Honest five features,0.526743
2,Deliberate label copy,1.000000


Leak score change relative to honest model: +0.473


In [8]:
X_leaky.drop(columns='leaked_label', inplace=True)
del X_leaky, leaky_model
X = data[FEATURE_COLS].copy()
final_model, final_score = fit_tree(X)
assert list(X.columns) == FEATURE_COLS and X.shape[1] == 5
assert 'leaked_label' not in X and 'outcome_clicks' not in X and 'is_declining' not in X
assert np.isclose(final_score, honest_score)
display(pd.DataFrame({'experiment': ['Final honest model after removing leak'],
                      'balanced_accuracy': [final_score]}))
print('The label copy supplied the answer. Removing it restores the honest score; no improvement is guaranteed.')

,experiment,balanced_accuracy
0,Final honest model after removing leak,0.526743


The label copy supplied the answer. Removing it restores the honest score; no improvement is guaranteed.


## 4. Data limits

**Main limitation: short-window noise.** Two 12-day windows can mistake normal variation,
including different weekday composition, for meaningful decline. A 20-click threshold
reduces some instability but does not eliminate it. No claim about seasonality or lasting
decline follows from this exercise.

- Requiring complete windows and minimum traffic excludes newer, low-volume, and poorly
  tracked pages. Complete outcome coverage is retrospective selection.
- Tracking start and non-null values cannot prove continuous measurement. A missing
  published search flag weakens the availability check further.
- A fixed snapshot cannot verify what each metric looked like historically before revisions.
  The three-day reporting buffer is an explicit assumption.
- Client holdout within March is not a future-month validation. This quick score does not
  establish final ranking performance, editorial time savings, or refresh benefit.
- The earlier click total appears in the label's denominator, so predicting the proxy may
  partly reflect its construction. Outcomes remain excluded from the honest predictors.

The notebook supports a measured decline-risk experiment. Human investigation and stronger
validation are needed before using a ranking to allocate editorial work.

## 5. Self-check

The following checks run only after every earlier cell succeeds. Successful execution
does not automatically commit, publish, or submit the notebook. Save outputs after a
fresh-kernel Run All, review the diff for credentials and private information, commit the
notebook and local setup files, and submit the repository URL. Never commit the cache.

In [9]:
assert len(FEATURE_COLS) == 5
assert 'IS TRUE' in availability_sql
assert FEATURE_END < DECISION_DATE <= OUTCOME_START
assert train_clients.isdisjoint(test_clients)
assert np.isclose(final_score, honest_score)
assert list(X.columns) == FEATURE_COLS
assert all('month=2026-03/' in p for p in manifest['source_files'])
checks = ['Five contract answers supplied', 'Three verification queries executed',
          'Availability filtered with IS TRUE', 'Five features with timing explanations',
          'Nonoverlapping feature and outcome windows', 'Client-disjoint evaluation',
          'Deliberate leak demonstrated and removed', 'Final honest score retained',
          'Limitations stated; June unused']
display(pd.DataFrame({'check': checks, 'passed': True}))
con.close()

,check,passed
0,Five contract answers supplied,True
1,Three verification queries executed,True
2,Availability filtered with IS TRUE,True
3,Five features with timing explanations,True
4,Nonoverlapping feature and outcome windows,True
5,Client-disjoint evaluation,True
6,Deliberate leak demonstrated and removed,True
7,Final honest score retained,True
8,Limitations stated; June unused,True
